# Day 20 Capstone: Image Classifier with Transfer Learning

Complete offline CIFAR-10 workflow: visualization, baseline CNN, MobileNetV2 feature extraction, fine-tuning, evaluation, Grad-CAM, and reflection.

## 0. Imports and offline balanced subset

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

RNG = 42
np.random.seed(RNG)
tf.random.set_seed(RNG)
print("TensorFlow:", tf.__version__)

# Reads the pre-seeded local Keras cache. Do not download manually if this fails.
(x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.cifar10.load_data()
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

def subset(x, y, per_class):
    y = y.ravel()
    rng = np.random.RandomState(RNG)
    idx = np.hstack([rng.choice(np.where(y == c)[0], per_class, replace=False) for c in range(10)])
    rng.shuffle(idx)
    return x[idx].astype("float32") / 255.0, y[idx].astype("int32")

x_train, y_train = subset(x_train_full, y_train_full, 500)
x_test, y_test = subset(x_test_full, y_test_full, 100)
print(x_train.shape, x_test.shape, "range", x_train.min(), x_train.max())
assert x_train.shape == (5000, 32, 32, 3)
assert x_test.shape == (1000, 32, 32, 3)

## 1. Visualize data and confirm balance

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(9, 4.7))
for ax, i in zip(axes.ravel(), range(18)):
    ax.imshow(x_train[i])
    ax.set_title(CLASSES[y_train[i]], fontsize=8)
    ax.axis("off")
plt.suptitle("CIFAR-10 subset — sample images")
plt.tight_layout()
plt.show()

counts = np.bincount(y_train, minlength=10)
for c, n in enumerate(counts):
    print(f"{CLASSES[c]:12s} {n}")
assert np.all(counts == 500)

## 2. Baseline CNN from scratch

In [ ]:
def make_baseline_cnn():
    model = keras.Sequential([
        keras.Input((32, 32, 3)),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax"),
    ], name="baseline_cnn")
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def plot_history(history, title):
    epochs = range(1, len(history.history["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    axes[0].plot(epochs, history.history["loss"], label="train")
    axes[0].plot(epochs, history.history["val_loss"], label="validation")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(epochs, history.history["accuracy"], label="train")
    axes[1].plot(epochs, history.history["val_accuracy"], label="validation")
    axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    fig.suptitle(title); plt.tight_layout(); plt.show()

baseline = make_baseline_cnn()
baseline.summary()
es = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)
hist_base = baseline.fit(x_train, y_train, validation_split=0.2, epochs=25, batch_size=64, callbacks=[es], verbose=2)
base_test = baseline.evaluate(x_test, y_test, verbose=0)[1]
print(f"BASELINE test accuracy = {base_test:.3f}")
plot_history(hist_base, "Baseline CNN learning curves")

## 3. Data augmentation

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.06),
    layers.RandomZoom(0.1),
], name="augment")

img = x_train[1][None, ...]
fig, axes = plt.subplots(1, 5, figsize=(11, 2.6))
axes[0].imshow(img[0]); axes[0].set_title("original", fontsize=9); axes[0].axis("off")
for ax in axes[1:]:
    ax.imshow(np.clip(data_augmentation(img, training=True)[0], 0, 1))
    ax.set_title("augmented", fontsize=9); ax.axis("off")
plt.suptitle(f"Augmentation preview — {CLASSES[y_train[1]]}")
plt.tight_layout(); plt.show()

## 4. Transfer learning: frozen MobileNetV2 feature extractor

In [ ]:
IMG = 96
base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input((32, 32, 3), name="image")
x = data_augmentation(inputs)
x = layers.Resizing(IMG, IMG, name="resize_for_mobilenet")(x)
x = layers.Rescaling(255.0, name="to_255")(x)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_average_pool")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation="softmax", name="classifier")(x)
tl_model = keras.Model(inputs, outputs, name="cifar10_mobilenetv2")
tl_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print(f"Trainable params (head only): {sum(int(np.prod(w.shape)) for w in tl_model.trainable_weights):,}")

ckpt = keras.callbacks.ModelCheckpoint("day20_best.keras", monitor="val_accuracy", save_best_only=True)
es2 = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)
hist_fe = tl_model.fit(x_train, y_train, validation_split=0.2, epochs=8, batch_size=64, callbacks=[es2, ckpt], verbose=2)
fe_test = tl_model.evaluate(x_test, y_test, verbose=0)[1]
print(f"FEATURE-EXTRACTION test accuracy = {fe_test:.3f}")
plot_history(hist_fe, "Frozen MobileNetV2 feature extraction")

## 5. Fine-tune the top 30 backbone layers

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
# Freezing BatchNormalization is safer for a small fine-tuning dataset.
for layer in base.layers[-30:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

tl_model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
es3 = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)
hist_ft = tl_model.fit(x_train, y_train, validation_split=0.2, epochs=6, batch_size=64, callbacks=[es3, ckpt], verbose=2)
ft_test = tl_model.evaluate(x_test, y_test, verbose=0)[1]
print(f"FINE-TUNED test accuracy = {ft_test:.3f}")
print(f"baseline={base_test:.3f} feat-extract={fe_test:.3f} fine-tuned={ft_test:.3f}")
plot_history(hist_ft, "MobileNetV2 fine-tuning")

## 6. Confusion matrix and per-class metrics

In [ ]:
y_prob = tl_model.predict(x_test, verbose=0)
y_pred = y_prob.argmax(axis=1)
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6.5, 6))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("Confusion matrix — fine-tuned model")
plt.tight_layout(); plt.show()
print(classification_report(y_test, y_pred, target_names=CLASSES, digits=3, zero_division=0))

cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
true_idx, pred_idx = np.unravel_index(np.argmax(cm_off), cm_off.shape)
worst_count = int(cm_off[true_idx, pred_idx])
worst_pair = f"{CLASSES[true_idx]} -> {CLASSES[pred_idx]}"
print(f"Worst directed confusion: {worst_pair} ({worst_count} test images)")

## 7. Grad-CAM

In [ ]:
# Create a helper model returning the nested backbone feature map and predictions.
backbone_layer = tl_model.get_layer(base.name)
grad_model = keras.Model(tl_model.inputs, [backbone_layer.output, tl_model.output])

def gradcam(img_1, pred_index=None):
    arr = tf.convert_to_tensor(img_1[None, ...], dtype=tf.float32)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(arr, training=False)
        pred_index = int(tf.argmax(preds[0]).numpy()) if pred_index is None else int(pred_index)
        score = preds[:, pred_index]
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.maximum(tf.reduce_sum(conv_out[0] * weights, axis=-1), 0)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return cam.numpy(), pred_index

correct = np.where(y_pred == y_test)[0]
show = correct[:3]

fig, axes = plt.subplots(2, 3, figsize=(9, 6))

for col, i in enumerate(show):
    cam, pidx = gradcam(x_test[i])

    cam_up = tf.image.resize(
        cam[..., None], (32, 32)
    ).numpy().squeeze()

    axes[0, col].imshow(x_test[i])
    axes[0, col].axis("off")
    axes[0, col].set_title(
        f"true: {CLASSES[y_test[i]]}\npred: {CLASSES[pidx]}",
        fontsize=9
    )

    axes[1, col].imshow(x_test[i])
    axes[1, col].imshow(cam_up, cmap="jet", alpha=0.45)
    axes[1, col].axis("off")
    axes[1, col].set_title("Grad-CAM", fontsize=9)

plt.suptitle("Where the model looks")
plt.tight_layout()
plt.show()

## 8. Results summary and reflection

In [ ]:
print("=== RESULTS SUMMARY ===")
print(f"{'approach':22s}{'test acc':>10s}")
for name, acc in [("Baseline CNN (scratch)", base_test), ("Transfer: feat-extract", fe_test), ("Transfer: fine-tuned", ft_test)]:
    print(f"{name:22s}{acc:10.3f}")
lift = ft_test - base_test
print(f"absolute lift (fine-tuned - baseline) = {lift:+.3f}")
print(f"worst confusion = {worst_pair} ({worst_count} images)")

# Read the Grad-CAM plots and edit the Grad-CAM sentence if your maps focus on background.
reflection = f"""
The fine-tuned transfer model reached {ft_test:.3f} test accuracy, compared with {base_test:.3f} for the CNN trained from scratch, an absolute improvement of {lift:+.3f}. Transfer learning helped because MobileNetV2 started with reusable ImageNet features such as edges, textures, and shapes instead of learning all visual features from only 5,000 low-resolution images. Frozen feature extraction reached {fe_test:.3f}, while fine-tuning changed accuracy by {ft_test - fe_test:+.3f}, showing the effect of gently adapting the top pretrained layers with a learning rate of 1e-5. The most frequent directed confusion was {worst_pair}, occurring for {worst_count} test images, which is understandable because similar CIFAR-10 objects lose detail at 32 by 32 pixels. In the displayed correct examples, the Grad-CAM maps should be checked to confirm that the strongest regions lie mainly on the object rather than on background cues. Next, I would try stronger label-preserving augmentation and compare another lightweight pretrained backbone while keeping the same validation and test protocol.
""".strip()
print("" + reflection)

## Final save and submission checks

In [ ]:
tl_model.save("day20_best.keras")
print("Saved model -> day20_best.keras")
print("Before submitting:")
print("1. Run all cells")
print("2. Confirm all plots are visible")
print("3. Edit the Grad-CAM sentence in Step 8 if needed")
print("4. Save as day20_capstone.ipynb")